# Test Registered Tapis MODFLOW Apps

This notebook logs into the `portals` tenant, inspects the registered apps, submits test jobs against the `ptdatax.project.PTDATAX-272` storage system, polls job status, and lists archived outputs.

Notes:
- `modflow-usg` uses the Carrizo-Wilcox USG model.
- `modflow-2000` uses the Yegua-Jackson MODFLOW-2000 model.
- `modflow-96` uses the Trinity Hill Country MODFLOW-96 model.
- `modflow6` is disabled by default below because the model inventory you provided does not include an obvious MF6 simulation directory or `mfsim.nam` path.

In [1]:
import json
import os
import time
from datetime import datetime
from getpass import getpass
from pathlib import Path
from pprint import pprint

import requests

requests.packages.urllib3.disable_warnings()

In [2]:
# Tenant and system configuration.
BASE_URL = "https://portals.tapis.io"
SYSTEM_ID = "ptdatax.project.PTDATAX-272"
MODEL_ROOT = "workingGAMs"

# Known registered app ids.
APP_CASES = {
    "modflow-usg-simulation": {
        "enabled": True,
        "description": "Carrizo-Wilcox MODFLOW-USG generated from package inputs",
        "file_inputs": {
            "mfusg-bas": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.bas",
            "mfusg-dis": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.dis",
            "mfusg-drn": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.drn",
            "mfusg-evt": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.evt",
            "mfusg-ghb": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.ghb",
            "mfusg-gnc": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.gnc",
            "mfusg-hfb": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.hfb",
            "mfusg-lpf": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.mod.lpf",
            "mfusg-oc": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.oc",
            "mfusg-rch": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.rch",
            "mfusg-riv": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.riv",
            "mfusg-sms": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.sms",
            "mfusg-wel": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.wel",
        },
        "expect": "SUCCEEDED",
    },
    "modflow-2000-simulation": {
        "enabled": True,
        "description": "Yegua-Jackson MODFLOW-2000 generated from package inputs",
        "file_inputs": {
            "mf2000-bas": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.bas",
            "mf2000-bcf": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.bcf",
            "mf2000-dis": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.dis",
            "mf2000-drn": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.drn",
            "mf2000-evt": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.evt",
            "mf2000-ghb": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.ghb",
            "mf2000-gmg": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.gmg",
            "mf2000-oc": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.oc",
            "mf2000-rch": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.rch",
            "mf2000-str": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.str",
            "mf2000-wel": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.wel",
        },
        "expect": "SUCCEEDED",
    },
    "modflow-96-simulation": {
        "enabled": True,
        "description": "Trinity Hill Country MODFLOW-96 generated from package inputs",
        "file_inputs": {
            "mf96-bas": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/bas.dat",
            "mf96-bcf": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/bcf.dat",
            "mf96-discret": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/discret.dat",
            "mf96-drn": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/drn.dat",
            "mf96-ghb": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/ghb.dat",
            "mf96-oc": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/oc.dat",
            "mf96-output": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/output.dat",
            "mf96-rch": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/rch.dat",
            "mf96-riv": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/riv.dat",
            "mf96-sor": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/sor.dat",
            "mf96-wel": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/wel.dat",
        },
        "expect": "FAILED until a real mf96 executable is added to the image",
    },
    "modflow6-simulation": {
        "enabled": False,
        "description": "MF6 test case not wired yet",
        "file_inputs": {
            # Example once you have a real MF6 simulation path on the shared system:
            # "mf6-sim-nam": f"{MODEL_ROOT}/some-mf6-model/mfsim.nam",
        },
        "expect": "Set a real MF6 model path before enabling",
    },
}

In [3]:
def tapis_url(path: str) -> str:
    path = path.lstrip("/")
    return f"tapis://{SYSTEM_ID}/{path}"


def get_token(username: str, password: str) -> str:
    response = requests.post(
        f"{BASE_URL}/v3/oauth2/tokens",
        json={"username": username, "password": password, "grant_type": "password"},
        timeout=60,
    )
    response.raise_for_status()
    payload = response.json()
    return payload["result"]["access_token"]["access_token"]


def api_headers(token: str) -> dict:
    return {
        "X-Tapis-Token": token,
        "Content-Type": "application/json",
    }


def api_get(token: str, path: str, **params):
    response = requests.get(
        f"{BASE_URL}{path}",
        headers=api_headers(token),
        params=params or None,
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["result"]


def api_post(token: str, path: str, body: dict):
    response = requests.post(
        f"{BASE_URL}{path}",
        headers=api_headers(token),
        data=json.dumps(body),
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["result"]

In [4]:
username = os.environ.get("TAPIS_USERNAME", "").strip() or input("Tapis username: ").strip()
password = os.environ.get("TAPIS_PASSWORD", "") or getpass("Tapis password: ")
token = get_token(username, password)
print(f"Authenticated as {username}")

Tapis username:  wmobley
Tapis password:  ········


Authenticated as wmobley


In [5]:
system_info = api_get(token, f"/v3/systems/{SYSTEM_ID}")
print("System:", system_info["id"])
print("Root dir:", system_info["rootDir"])
print("Shared with:", system_info.get("sharedWithUsers", []))

System: ptdatax.project.PTDATAX-272
Root dir: /corral-repl/tacc/aci/PT2050/projects/PTDATAX-272
Shared with: ['wma_prtl', 'wmobley', 'sawp33', 'lpearson']


In [6]:
def get_app(app_id: str) -> dict:
    return api_get(token, f"/v3/apps/{app_id}")


def get_named_file_inputs(app_def: dict) -> dict:
    job_attrs = app_def.get("jobAttributes", {})
    return {item["name"]: item for item in job_attrs.get("fileInputs", [])}


def build_job_file_inputs(app_def: dict, source_map: dict) -> list[dict]:
    named_inputs = get_named_file_inputs(app_def)
    file_inputs = []
    for name, rel_path in source_map.items():
        if name not in named_inputs:
            raise KeyError(f"Input {name!r} is not defined on app {app_def['id']}")
        file_inputs.append(
            {
                "name": name,
                "sourceUrl": tapis_url(rel_path),
            }
        )
    return file_inputs


def make_job_body(app_def: dict, case: dict) -> dict:
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    body = {
        "name": f"test-{app_def['id']}-{timestamp}",
        "appId": app_def["id"],
        "appVersion": app_def["version"],
    }
    file_inputs = build_job_file_inputs(app_def, case.get("file_inputs", {}))
    if file_inputs:
        body["fileInputs"] = file_inputs
    return body


def submit_job(body: dict) -> dict:
    return api_post(token, "/v3/jobs/submit", body)


def get_job(job_uuid: str) -> dict:
    return api_get(token, f"/v3/jobs/{job_uuid}")


def get_job_status(job_uuid: str) -> dict:
    return api_get(token, f"/v3/jobs/{job_uuid}/status")


def api_get_optional(token: str, path: str, **params):
    response = requests.get(
        f"{BASE_URL}{path}",
        headers=api_headers(token),
        params=params or None,
        timeout=60,
    )
    if response.status_code >= 400:
        return {"ok": False, "status_code": response.status_code, "text": response.text}
    payload = response.json()
    return {"ok": True, "result": payload.get("result")}


def get_job_history(job_uuid: str) -> dict:
    candidate_paths = [
        f"/v3/jobs/{job_uuid}/history",
        f"/v3/jobs/{job_uuid}/events",
        f"/v3/jobs/{job_uuid}/messages",
    ]
    attempts = []
    for path in candidate_paths:
        result = api_get_optional(token, path)
        attempts.append({"path": path, **result})
        if result.get("ok"):
            return {"path": path, "result": result.get("result"), "attempts": attempts}
    return {"path": None, "result": None, "attempts": attempts}


def list_outputs(job_uuid: str, output_path: str = "") -> list:
    suffix = f"/{output_path.lstrip('/')}" if output_path else "/"
    return api_get(token, f"/v3/jobs/{job_uuid}/output/list{suffix}")


TERMINAL_STATES = {"FINISHED", "FAILED", "CANCELLED", "STOPPED"}


def wait_for_job(job_uuid: str, poll_seconds: int = 20, max_polls: int = 120) -> dict:
    for attempt in range(1, max_polls + 1):
        status = get_job_status(job_uuid)
        state = status.get("status") or status.get("state")
        print(f"[{attempt:03d}] {job_uuid}: {state}")
        if state in TERMINAL_STATES:
            return status
        time.sleep(poll_seconds)
    raise TimeoutError(f"Job {job_uuid} did not reach a terminal state in time")

In [7]:
app_catalog = {}
for app_id, case in APP_CASES.items():
    app_def = get_app(app_id)
    app_catalog[app_id] = app_def
    print(f"\n{app_id}")
    print(f"  version: {app_def['version']}")
    print(f"  image:   {app_def['containerImage']}")
    print(f"  enabled: {case['enabled']}")
    print(f"  expect:  {case['expect']}")
    print("  file inputs:")
    for item in app_def.get("jobAttributes", {}).get("fileInputs", []):
        print(f"    - {item['name']}: {item['targetPath']}")


modflow-usg-simulation
  version: 0.0.1
  image:   docker://ghcr.io/wmobley/modflow-usg:0.0.1
  enabled: True
  expect:  SUCCEEDED
  file inputs:
    - mfusg-simulation-archive: simulation.zip
    - mfusg-nam: provided/model.nam
    - mfusg-bas: provided/model.bas
    - mfusg-dis: provided/model.dis
    - mfusg-drn: provided/model.drn
    - mfusg-evt: provided/model.evt
    - mfusg-ghb: provided/model.ghb
    - mfusg-gnc: provided/model.gnc
    - mfusg-hfb: provided/model.hfb
    - mfusg-lpf: provided/model.lpf
    - mfusg-oc: provided/model.oc
    - mfusg-rch: provided/model.rch
    - mfusg-riv: provided/model.riv
    - mfusg-sms: provided/model.sms
    - mfusg-wel: provided/model.wel

modflow-2000-simulation
  version: 0.0.1
  image:   docker://ghcr.io/wmobley/modflow-2000:0.0.1
  enabled: True
  expect:  SUCCEEDED
  file inputs:
    - mf2000-simulation-archive: simulation.zip
    - mf2000-nam: provided/model.nam
    - mf2000-bas: provided/model.bas
    - mf2000-bcf: provided/model.

In [8]:
def run_case(app_id: str, case: dict, wait: bool = True) -> dict:
    app_def = app_catalog[app_id]
    body = make_job_body(app_def, case)
    print("Submitting job body:")
    pprint(body)
    job = submit_job(body)
    job_uuid = job["uuid"]
    print(f"Submitted {app_id} -> {job_uuid}")
    result = {"app_id": app_id, "job_uuid": job_uuid, "submit": job}
    if wait:
        status = wait_for_job(job_uuid)
        result["status"] = status
        try:
            result["outputs"] = list_outputs(job_uuid)
        except Exception as exc:
            result["outputs_error"] = str(exc)
    return result

In [9]:
# Run all enabled test cases.
results = {}
for app_id, case in APP_CASES.items():
    if not case.get("enabled", False):
        print(f"Skipping {app_id}: {case['expect']}")
        continue
    print("\n" + "=" * 80)
    print(f"Running {app_id}: {case['description']}")
    results[app_id] = run_case(app_id, case, wait=True)


Running modflow-usg-simulation: Carrizo-Wilcox MODFLOW-USG
Submitting job body:
{'appId': 'modflow-usg-simulation',
 'appVersion': '0.0.1',
 'fileInputs': [{'autoMountLocal': True,
                 'name': 'mfusg-nam',
                 'sourceUrl': 'tapis://ptdatax.project.PTDATAX-272/workingGAMs/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.mod.nam',
                 'targetPath': 'provided/model.nam'}],
 'name': 'test-modflow-usg-simulation-20260429-141710'}
Submitted modflow-usg-simulation -> 29dc9ea9-d5a2-4e5b-a862-b59cfe84342b-007
[001] 29dc9ea9-d5a2-4e5b-a862-b59cfe84342b-007: PENDING
[002] 29dc9ea9-d5a2-4e5b-a862-b59cfe84342b-007: STAGING_INPUTS
[003] 29dc9ea9-d5a2-4e5b-a862-b59cfe84342b-007: STAGING_INPUTS
[004] 29dc9ea9-d5a2-4e5b-a862-b59cfe84342b-007: FAILED

Running modflow-2000-simulation: Yegua-Jackson MODFLOW-2000
Submitting job body:
{'appId': 'modflow-2000-simulation',
 'appVersion': '0.0.1',
 'fileInputs': [{'autoMountLocal': True,
                 'name': 'mf

In [10]:
# Summarize final states.
summary = {}
for app_id, result in results.items():
    status = result.get("status", {})
    summary[app_id] = {
        "job_uuid": result["job_uuid"],
        "state": status.get("status") or status.get("state"),
        "expect": APP_CASES[app_id]["expect"],
    }
summary

{'modflow-usg-simulation': {'job_uuid': '29dc9ea9-d5a2-4e5b-a862-b59cfe84342b-007',
  'state': 'FAILED',
  'expect': 'SUCCEEDED'},
 'modflow-2000-simulation': {'job_uuid': '6451992a-c078-40e6-bd7f-7978eb88bf02-007',
  'state': 'FAILED',
  'expect': 'SUCCEEDED'},
 'modflow-96-simulation': {'job_uuid': '4e3994e9-f6a7-4531-901a-d4cedd0325a4-007',
  'state': 'FAILED',
  'expect': 'FAILED until a real mf96 executable is added to the image'}}

In [ ]:
# Inspect one job in more detail, including archived output listing.
APP_ID_TO_INSPECT = "modflow-usg-simulation"

job_uuid = results[APP_ID_TO_INSPECT]["job_uuid"]
job_detail = get_job(job_uuid)
output_listing = list_outputs(job_uuid)

print("Job detail:")
pprint(job_detail)
print("\nArchived outputs:")
for item in output_listing:
    print(f"- {item.get('path', item.get('name'))}")

In [ ]:
# Diagnose failed jobs by querying job detail, status, history/messages, and archived outputs.
FAILED_JOB_UUIDS = [
    "29dc9ea9-d5a2-4e5b-a862-b59cfe84342b-007",
    "6451992a-c078-40e6-bd7f-7978eb88bf02-007",
    "4e3994e9-f6a7-4531-901a-d4cedd0325a4-007",
]


def print_job_diagnostics(job_uuid: str) -> None:
    print("\n" + "=" * 100)
    print(f"Job: {job_uuid}")

    detail = get_job(job_uuid)
    status = get_job_status(job_uuid)
    history = get_job_history(job_uuid)

    print("\nJob detail:")
    pprint(detail)

    print("\nJob status:")
    pprint(status)

    print("\nHistory/messages endpoint:", history["path"])
    if history["result"] is not None:
        pprint(history["result"])
    else:
        print("No history endpoint succeeded. Attempts:")
        pprint(history["attempts"])

    print("\nArchived outputs:")
    try:
        for item in list_outputs(job_uuid):
            print(f"- {item.get('path', item.get('name'))}")
    except Exception as exc:
        print(f"Unable to list outputs: {exc}")


for job_uuid in FAILED_JOB_UUIDS:
    print_job_diagnostics(job_uuid)
